# 04 — Current team state: form, momentum, stability

Trailing-window form vs season baseline, activity gaps (rust), patch splits,
and a manual roster-news overlay (fill from Liquipedia/research — roster
changes are the #1 model blind spot).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)
DATA = ROOT / 'data'


In [ ]:
matches = pd.read_parquet(DATA / 'matches.parquet')
teams = pd.read_parquet(DATA / 'teams.parquet')[['team_id','name']]
m = matches.dropna(subset=['radiant_team_id','dire_team_id']).copy()
now = m.start_time.max()
long = pd.concat([
    m.assign(team_id=m.radiant_team_id, win=m.radiant_win),
    m.assign(team_id=m.dire_team_id, win=~m.radiant_win.astype(bool)),
])
def wr(days):
    w = long[long.start_time >= now - days*86400]
    return w.groupby('team_id').win.agg(['mean','size']).rename(
        columns={'mean': f'wr_{days}d', 'size': f'n_{days}d'})
form = wr(30).join(wr(60), how='outer').join(wr(180), how='outer')
form = form.join(long.groupby('team_id').start_time.max().rename('last_game'))
form['days_idle'] = (now - form.last_game) / 86400
form = form.reset_index().merge(teams, on='team_id')
form['momentum'] = form.wr_30d - form.wr_180d   # + = improving
form.sort_values('wr_60d', ascending=False).head(20)

In [ ]:
# Patch adaptability: winrate on the current patch vs previous
cur = m.patch.max()
split = pd.concat([
    long[long.patch==cur].groupby('team_id').win.mean().rename('wr_cur_patch'),
    long[long.patch==cur-1].groupby('team_id').win.mean().rename('wr_prev_patch'),
], axis=1).reset_index().merge(teams, on='team_id')
split['patch_delta'] = split.wr_cur_patch - split.wr_prev_patch
split.dropna().sort_values('patch_delta', ascending=False).head(15)

In [ ]:
# Manual overlay: roster changes / standins / red flags since TI qualifiers.
# Fill from research (research/TI2026_UNDERVALUED.md) and Liquipedia.
ROSTER_NOTES = {
    # 'Team Spirit': 'stable since TI25; no changes',
    # 'Tundra Esports': 'new pos5 May 2026 - cohesion risk',
}
state = form.sort_values('wr_60d', ascending=False).head(16).copy()
state['roster_note'] = state.name.map(ROSTER_NOTES).fillna('(fill me)')
state[['name','wr_30d','n_30d','wr_60d','n_60d','momentum','days_idle','roster_note']]